# Simulation Code T = 1

## Import libraries

In [1]:
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.dynamicRieszFunctions import estimateDynamicRiesz_all
from utils.dynamicRieszBradic import estimateBradicT1
import torch
import pandas as pd
import time
import numpy as np
from torch.distributions import Normal


/Users/wisserutgers/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Estimation Settings

In [2]:
lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : 0.1, # CHANGED FROM "CV"
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : 0.1, # CHANGED FROM "CV"
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

rf_f_settings = {
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' : -1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}


## Generate data 

#### Simulation settings:

In [3]:
torch.manual_seed(123);

# Parameters
N = 5000
tmax = 20

# Higher dimensions = more sparsity. Minimum is dimX1 = dimX2 = 3
p1 = 1

#### Define treatment probability function

In [4]:
# Bounds (only for truncated distributions)
lower = 0.10
upper = 0.90

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

# Define func_link function (nonlinear probability function from Bradic)
def func_link(x):
    return (x > 0) * torch.abs(x) / (torch.abs(x) + 1) + (x < 0) / (torch.abs(x) + 1)

# Define a truncated func_link function
def truncated_link(x):
    return lower + (upper - lower) * func_link(x)

# Simple nonlinear probability (from adversarial Riesz paper)
def truncated_adv(x):
    return lower + (upper - lower) * (x > 0)

# Simple nonlinear probability (from adversarial Riesz paper)
def double_nonlinear(x):
    return lower + (upper - lower) * (x > - 1/2) * (x < 1/2)

# Simple nonlinear probability (from adversarial Riesz paper)
def nonsymmetric(x):
    return lower + 0.5 * (upper - lower) * (x < - 1/4) + (upper - lower) * (x > 1/4) 

normal_dist = Normal(0, 1)
def probit(x):
    return lower + (upper - lower) * normal_dist.cdf(x)


In [5]:
treatment_probability_func = truncated_adv

#### Generate data

In [6]:
# Coefficients
# beta_pi1_0 = 0
# beta_pi1_S1 = torch.tensor([1, 1,0,0,0,0], dtype=torch.float32).reshape(-1,1) 
# beta_g0_0 = 1 * 4
# beta_g0_S1 = torch.tensor([1, 0,0,0,0,0], dtype=torch.float32).reshape(-1,1) * 2
# beta_g1_0 = -1 * 5
# beta_g1_S1 = torch.tensor([-1, 0,0,0,0,0], dtype=torch.float32).reshape(-1,1) * 1

# # 1 covariate
# beta_pi1_S1 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) 
# beta_g0_0 = 2 * 1
# beta_g0_S1 = torch.tensor([-1], dtype=torch.float32).reshape(-1,1) * 6
# beta_g1_0 = -1 * 3
# beta_g1_S1 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) * 9

# Adversarial Riesz paper
beta_pi1_0 = 0
beta_pi1_S1 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) 
beta_g0_0 = 0
beta_g0_S1 = torch.tensor([1.2], dtype=torch.float32).reshape(-1,1) 
beta_g1_0 = 2.2
beta_g1_S1 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) 

# Set seed
torch.manual_seed(123)

# Generate data
S1_all = torch.randn(N * tmax, p1)
pi1_all = treatment_probability_func(beta_pi1_0 + S1_all @ beta_pi1_S1).reshape(-1,1)
A1_all = torch.bernoulli(pi1_all).int().reshape(-1,1)

# g_all = ((A1_all == 0).float() * (beta_g0_0 + S1_all @ beta_g0_S1 ) + 
#          (A1_all == 1).float() * (beta_g1_0 + S1_all @ beta_g1_S1 ))
# g_all = ((A1_all == 0).float() * (beta_g0_0 + (S1_all @ beta_g0_S1 > 0) * 3 ) + 
#          (A1_all == 1).float() * (beta_g1_0 + (S1_all @ beta_g1_S1> 0) * 2 ))
g_all = ((A1_all == 0).float() * (beta_g0_0 + (S1_all > 0) * beta_g0_S1 ) + 
         (A1_all == 1).float() * (beta_g1_0 + (S1_all > 0) * beta_g0_S1 + S1_all ))
zeta_all = torch.randn(N * tmax,1)
Y_all = g_all + zeta_all

#### True values:

In [7]:
# mu1_1_all = beta_g1_0 + (S1_all @ beta_g0_S1 > 0) * 3
# mu1_0_all = beta_g0_0 + (S1_all @ beta_g1_S1 > 0) * 2

# theta0 = beta_g0_0 + 0.5 * 3
# theta1 = beta_g1_0 + 0.5 * 2

# mu1_1_all = beta_g1_0 + S1_all @ beta_g1_S1
# mu1_0_all = beta_g0_0 + S1_all @ beta_g0_S1

# theta0 = beta_g0_0 
# theta1 = beta_g1_0 

mu1_1_all = beta_g1_0 + S1_all + (S1_all > 0) * beta_g0_S1
mu1_0_all = beta_g0_0 + (S1_all > 0) * beta_g0_S1

theta0 = beta_g0_0 + 0.5 * 1.2
theta1 = beta_g1_0 + 0.5 * 1.2 

theta = theta1 - theta0
print(theta, theta1, theta0)


2.2 2.8000000000000003 0.6


## Estimation:

#### Estimation settings

In [8]:
folds = 2

time0 = time.time()

#### Estimation 

In [9]:
pred_theta = torch.zeros(tmax,5)
pred_sig = torch.zeros(tmax,5)

RR1 = torch.zeros(N,tmax,5)
RR2 = torch.zeros(N,tmax,5)
f1 = torch.zeros(N,tmax,5)
f2 = torch.zeros(N,tmax,5)

for t in range(0,tmax):

    # Get data for current iteration
    S1 = S1_all[(t) * N : (t+1) * N, :]
    A1 = A1_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]

    X = ((S1))
    X_index = torch.tensor([S1.shape[1]-1])
    D = ((A1))

    # Set counterfactual of interest
    d = 1 * torch.ones(D.shape) 

    #---------------------------------------------------------------------------
    # Estimator 1: Oracle
    pi1 = pi1_all[(t) * N : (t+1) * N]
    mu1_1 = mu1_1_all[(t) * N : (t+1) * N]
    mu1_0 = mu1_0_all[(t) * N : (t+1) * N]
    if d[0] == 1:
        gamma = A1 / pi1  
        theta_hat = (gamma * Y - (gamma - 1) * mu1_1)
        pred_theta[t,0] = torch.mean( theta_hat )
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
        f2[:,t,:1] = mu1_1
    else:
        gamma = (1 - A1) / (1 - pi1  )
        theta_hat = (gamma * Y - (gamma - 1) * mu1_0  )
        pred_theta[t,0] = torch.mean( theta_hat )
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
        f2[:,t,:1] = mu1_0

    RR2[:,t,:1] = gamma

    #---------------------------------------------------------------------------
    # Estimator 2: Bradic
    bradic_result = estimateBradicT1(Y,S1,A1, folds)
    if d[0] == 1:
        pred_theta[t,1] = bradic_result[1]
        pred_sig[t,1] = bradic_result[3]
        RR2[:,t,1:2] = bradic_result[5]
        f2[:,t,1:2] = bradic_result[7]
    elif d[0] == 0:
        pred_theta[t,1] = bradic_result[2]
        pred_sig[t,1] = bradic_result[4]
        RR2[:,t,1:2] = bradic_result[6]
        f2[:,t,1:2] = bradic_result[8]

    coeff = bradic_result[7]
    #---------------------------------------------------------------------------
    # Estimator 3: LASSO 

    # pred_theta[t,2], pred_sig[t,2], RR1[:,t,2:3], RR2[:,t,2:3], f1[:,t,2:3], f2[:,t,2:3] = estimateDynamicRiesz(Y, X, D, d, X_index, folds,
    #                                                                  method_a = "LASSO", lasso_a_settings = lasso_a_settings,
    #                                                                     method_f = "LASSO", lasso_f_settings = lasso_f_settings)

    # #---------------------------------------------------------------------------
    # # Estimator 4: RF 

    # pred_theta[t,3], pred_sig[t,3], RR1[:,t,3:4], RR2[:,t,3:4], f1[:,t,3:4], f2[:,t,3:4] = estimateDynamicRiesz(Y, X, D, d, X_index, folds,
    #                                                                  method_a = "RF", rf_a_settings = rf_a_settings,
    #                                                                     method_f = "RF", rf_f_settings = rf_f_settings)

    # # # #---------------------------------------------------------------------------
    # # # # Estimator 5: Net 

    # pred_theta[t,4], pred_sig[t,4], RR1[:,t,4:5], RR2[:,t,4:5],  f1[:,t,4:5], f2[:,t,4:5] = estimateDynamicRiesz(Y, X, D, d, X_index, folds,
    #                                                                  method_a = "Net", net_a_settings = net_a_settings,
    #                                                                     method_f = "Net", net_f_settings = net_f_settings)

    pred_theta[t,2:], pred_sig[t,2:], RR1[:,t,2:], RR2[:,t,2:], f1[:,t,2:], f2[:,t,2:] = estimateDynamicRiesz_all(Y, X, D, d, X_index, folds)

    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


tensor([[0.2539],
        [0.2536],
        [2.8317],
        [1.0163]])
tensor([[0.3175],
        [0.3192],
        [2.8273],
        [0.9538]])
Time until iteration  0 :  3.4968440532684326
tensor([[0.3052],
        [0.2851],
        [2.8250],
        [0.9765]])
tensor([[0.3452],
        [0.2934],
        [2.6503],
        [1.1000]])
tensor([[0.3481],
        [0.3386],
        [2.7770],
        [0.9489]])
tensor([[0.3525],
        [0.3062],
        [2.7200],
        [1.0061]])
tensor([[0.2853],
        [0.3159],
        [2.8017],
        [0.9888]])
tensor([[0.3622],
        [0.3412],
        [2.7403],
        [0.9912]])
tensor([[0.3143],
        [0.2793],
        [2.7919],
        [1.0119]])
tensor([[0.3141],
        [0.2777],
        [2.7365],
        [0.9904]])
tensor([[0.3561],
        [0.3256],
        [2.6710],
        [0.9721]])
tensor([[0.4003],
        [0.3619],
        [2.6905],
        [0.9140]])
tensor([[0.3481],
        [0.3017],
        [2.7370],
        [0.9541]])
tenso

## Compute statistics

#### Get true value

In [10]:
true_value = theta1

#### Compute statistics

In [11]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["Oracle", "Bradic", "LASSO", "RF", "Net"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

#### Print results

In [12]:
df = pd.DataFrame(data)
print(df)

   Method      Bias      RMSE  Coverage  Interval Length
0  Oracle -0.007654  0.036877      0.95         0.156062
1  Bradic  0.267390  0.722790      0.95         2.205797
2   LASSO  0.282319  0.283708      0.00         0.136941
3      RF -0.022785  0.047870      0.95         0.185130
4     Net -2.800000  2.800000      0.00         0.000000


In [13]:
pred_theta

tensor([[2.8347, 2.9324, 3.1136, 2.8004, 0.0000],
        [2.7639, 2.3938, 3.0556, 2.7492, 0.0000],
        [2.8009, 2.7336, 3.1174, 2.8116, 0.0000],
        [2.7985, 2.1584, 3.1006, 2.8008, 0.0000],
        [2.7395, 3.0758, 3.0472, 2.7187, 0.0000],
        [2.7832, 5.4950, 3.0488, 2.7516, 0.0000],
        [2.8244, 2.8450, 3.0887, 2.8084, 0.0000],
        [2.7590, 3.8717, 3.0699, 2.7250, 0.0000],
        [2.7772, 3.5758, 3.1059, 2.7534, 0.0000],
        [2.8123, 3.0330, 3.0930, 2.7807, 0.0000],
        [2.8263, 3.1281, 3.0818, 2.8166, 0.0000],
        [2.7973, 2.6220, 3.1039, 2.7703, 0.0000],
        [2.7903, 2.6855, 3.1186, 2.7887, 0.0000],
        [2.7538, 3.4444, 3.0373, 2.7504, 0.0000],
        [2.7771, 2.9809, 3.0700, 2.7822, 0.0000],
        [2.7880, 2.8311, 3.0606, 2.7764, 0.0000],
        [2.7204, 2.6601, 3.0406, 2.6787, 0.0000],
        [2.7840, 2.9655, 3.0782, 2.7836, 0.0000],
        [2.8793, 2.9026, 3.1373, 2.8766, 0.0000],
        [2.8367, 3.0131, 3.0774, 2.8210, 0.0000]])

In [14]:
check_t = 0
# Compare RR:
pd.DataFrame(RR2[:,check_t,:].numpy().squeeze()).head(50)

,0,1,2,3,4
0,1.111111,1.450398,2.477577,1.161804,0.0
1,0.000000,0.000000,0.033516,0.000000,0.0
2,0.000000,0.000000,0.033516,0.000000,0.0
3,0.000000,0.000000,0.033516,0.000000,0.0
4,1.111111,1.440606,2.422950,1.150487,0.0
5,1.111111,1.224803,1.833463,1.146133,0.0
6,10.000000,2.523682,3.528239,13.963461,0.0
7,0.000000,0.000000,0.004227,0.000000,0.0
8,1.111111,1.175889,1.666981,1.086500,0.0
9,0.000000,0.000000,0.033516,0.000000,0.0


In [15]:
pd.DataFrame(f2[:,check_t,:].numpy().squeeze()).head(50)

,0,1,2,3,4
0,3.737370,3.517396,3.514087,3.514087,0.0
1,2.022223,2.920760,2.918522,2.918522,0.0
2,1.896472,2.760625,2.758443,2.758443,0.0
3,1.611988,2.398353,2.396296,2.396296,0.0
4,3.748605,3.591073,3.588605,3.588605,0.0
5,4.060341,3.988047,3.985443,3.985443,0.0
6,1.980362,2.811453,2.806756,2.806756,0.0
7,1.820830,2.609264,2.604169,2.604169,0.0
8,4.167107,4.062038,4.059800,4.059800,0.0
9,1.007498,1.628576,1.626783,1.626783,0.0


### RR MSE

In [16]:
torch.sqrt(torch.mean( torch.mean( (RR2[:,:,:1] - RR2[:,:,1:]) ** 2,0),0))

tensor([68.1122,  1.4779,  0.5767,  2.3675])

### regresion MSE

In [17]:
torch.sqrt(torch.mean( torch.mean( (f2[:,:,:1] - f2[:,:,1:]) ** 2,0),0))

tensor([0.5036, 0.5012, 0.5012, 3.1795])